In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

In [ ]:
df = pd.read_csv("data/train.csv")
df = df.drop(columns=["PassengerId", "Name", "Ticket", "Cabin", "SibSp"])

# ---- фичи / таргет ----
X = df.drop(columns="Survived")
y = df["Survived"]

num_cols = ["Age", "Fare", "Parch", "Pclass"]
cat_cols = ["Sex", "Embarked"]

# ---- препроцессинг ----
numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler",  StandardScaler()),
])
categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot",  OneHotEncoder(handle_unknown="ignore", drop="first")),
])

preprocess = ColumnTransformer([
    ("num", numeric_pipe, num_cols),
    ("cat", categorical_pipe, cat_cols),
])

# ---- train/test split (ДО обучения, стратифицируем по y) ----
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ---- модель ----
model = Pipeline([
    ("prep", preprocess),
    ("clf",  LogisticRegression(max_iter=1000, random_state=42, penalty='elasticnet', solver='saga')),
])
model.fit(X_train, y_train)

# ---- оценка ----
y_pred  = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred, digits=3))
print("ROC-AUC:", round(roc_auc_score(y_test, y_proba), 3))
print(confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

           0      0.802     0.845     0.823       110
           1      0.730     0.667     0.697        69

    accuracy                          0.777       179
   macro avg      0.766     0.756     0.760       179
weighted avg      0.774     0.777     0.774       179

ROC-AUC: 0.829


c:\Users\АуфПК\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
